# 🇪🇹 Ethiopia — Climate Data Profiling, Cleaning & EDA

**Project:** EthioClimate Analytics — COP32 Position Paper Support  
**Branch:** `eda-ethiopia`  
**Data Source:** NASA POWER (2015–2026)  

---

## Notebook Objectives
1. Load, parse dates, and profile the raw Ethiopia dataset
2. Replace NASA sentinel values (`-999`) and handle missing data
3. Detect and document outliers
4. Conduct time-series, correlation, and distribution analysis
5. Export a cleaned CSV to `data/sudan_clean.csv`

---

### 🪜 The Negotiation-Grade Ladder
Every chart in this notebook aims to answer:
| Layer | Question | Status |
|-------|----------|---------|
| 1 | What is **changing**? (trend + baseline) | ✅ EDA |
| 2 | What did it **cause**? (impact stat) | ✅ Report |
| 3 | What does it **demand**? (policy ask) | ✅ Position Paper |

## 0. Setup & Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# Our reusable modules
from src.clean import load_and_parse, report_missing, flag_outliers, clean_dataframe, export_clean
from src.visualize import (
    plot_monthly_temperature, plot_monthly_precipitation,
    plot_correlation_heatmap, plot_scatter_t2m_rh2m,
    plot_precipitation_histogram, plot_bubble_chart
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

COUNTRY = 'Sudan'
RAW_PATH = '../data/sudan.csv'
CLEAN_PATH = '../data/sudan_clean.csv'

print('✅ Imports successful')

---
## 1. Data Loading & Date Parsing

We use `YEAR` and `DOY` (day-of-year) to build a proper `datetime` column.  
**Why?** NASA POWER delivers data in this format so we can reconstruct exact dates without timezone ambiguity.

In [ ]:
df = load_and_parse(RAW_PATH, COUNTRY)

print(f'Shape: {df.shape}')
print(f'Date range: {df["Date"].min()} → {df["Date"].max()}')
df.head()

In [ ]:
# Check column dtypes
df.dtypes

---
## 2. Summary Statistics & Missing-Value Report

> **Note:** `-999` values were already replaced with `NaN` in `load_and_parse()`.  
> This is NASA's sentinel for missing or out-of-range data. Running statistics on `-999` would completely distort means and standard deviations.

In [ ]:
# Summary statistics
df.describe()

**📝 Interpretation (fill in after running):**
- `T2M` mean: __ °C — typical for Ethiopian highland/lowland mix
- `PRECTOTCORR` is highly right-skewed (most days have low or zero rain)
- `RH2M` ranges from __ to __ — reflects wet/dry season contrast
- `T2M_RANGE` reveals diurnal temperature variation — higher in dry season

In [ ]:
# Duplicate rows
n_dup = df.duplicated().sum()
print(f'Duplicate rows found: {n_dup}')
if n_dup > 0:
    df = df.drop_duplicates()
    print(f'Duplicates dropped. New shape: {df.shape}')

In [ ]:
# Missing value report
missing_report = report_missing(df)
print('Columns with missing values:')
display(missing_report)

# Flag columns with >5% missing
high_missing = missing_report[missing_report['missing_pct'] > 5]
if not high_missing.empty:
    print('\n⚠️  Columns with >5% missing values:')
    display(high_missing)
else:
    print('\n✅ No columns exceed the 5% missing threshold.')

**📝 Missing Data Notes:**  
_(Fill in after running — e.g., "PS has X% missing, likely due to altitude-related sensor gaps at the representative station.")_

---
## 3. Outlier Detection & Cleaning

We use **Z-scores** to detect statistical outliers (|Z| > 3 means the value is more than 3 standard deviations from the mean — extremely rare in a normal distribution).

**Decision rule:** We will **retain** outliers that correspond to known extreme weather events (they are real data!) but **document** them clearly. We cap only clear instrument errors.

In [ ]:
WEATHER_COLS = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']

outlier_mask = flag_outliers(df, WEATHER_COLS, z_threshold=3.0)
print(f'Outlier rows detected (|Z| > 3): {outlier_mask.sum()} / {len(df)} ({outlier_mask.sum()/len(df)*100:.2f}%)')

# Preview outlier rows
df[outlier_mask][['Date', 'T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M']].head(20)

**📝 Outlier Decision:**  
We choose to **retain** outlier rows because:
- Ethiopia experiences genuine extreme rainfall events (ITCZ-driven)
- Extreme heat days are real climate signal, not instrument noise
- Dropping them would undercount extreme events — the opposite of what a COP32 position paper needs

In [ ]:
# Full cleaning pipeline
df_clean = clean_dataframe(
    df,
    outlier_cols=WEATHER_COLS,
    z_threshold=3.0,
    outlier_action='retain',  # keep real extreme events
    fill_method='ffill',
    missing_row_threshold=0.30
)

print(f'\nFinal cleaned DataFrame shape: {df_clean.shape}')
df_clean.head()

In [ ]:
# Confirm no remaining NaNs in key columns
print('Remaining NaNs after cleaning:')
df_clean[WEATHER_COLS].isna().sum()

In [ ]:
# Export cleaned data
export_clean(df_clean, CLEAN_PATH)

---
## 4. Time Series Analysis

### 4.1 Monthly Average Temperature (2015–2026)

**What are we looking for?**  
A rising trend in `T2M` over the 11-year window is the primary climate signal. We're also looking for the warmest and coolest anomalies which may correspond to El Niño / La Niña years.

In [ ]:
fig = plot_monthly_temperature(df_clean, COUNTRY)
plt.show()

**📝 Trend Interpretation:**  
_(Fill in after running — e.g., "There is a visible upward trend of approximately X°C per decade. The warmest month was [month] during the [El Niño / La Niña] event of [year].")_

**🌍 COP32 Relevance:**  
Ethiopia sits at the heart of the East African warming hotspot. A rising temperature trend directly threatens the country's rain-fed agriculture sector which employs ~70% of the population.

### 4.2 Monthly Total Precipitation

In [ ]:
fig = plot_monthly_precipitation(df_clean, COUNTRY)
plt.show()

**📝 Precipitation Pattern Interpretation:**  
_(Fill in — e.g., "Ethiopia shows a bimodal rainfall pattern: the short rains (Belg) from Feb–May and the main rains (Kiremt) from Jun–Sep. The [year] season shows anomalously low totals consistent with the 2020–2023 Horn of Africa drought.")_

---
## 5. Correlation & Relationship Analysis

### 5.1 Correlation Heatmap

In [ ]:
fig = plot_correlation_heatmap(df_clean, COUNTRY)
plt.show()

**📝 Top 3 Correlations:**  
_(Fill in after running)_
1. **__ vs __** (r = __): _interpretation_
2. **__ vs __** (r = __): _interpretation_
3. **__ vs __** (r = __): _interpretation_

### 5.2 Scatter Plots

In [ ]:
# T2M vs RH2M
fig = plot_scatter_t2m_rh2m(df_clean, COUNTRY)
plt.show()

In [ ]:
# T2M_RANGE vs WS2M
from src.visualize import set_style
set_style()
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df_clean['T2M_RANGE'], df_clean['WS2M'], alpha=0.3, s=10, color='#f97316')
ax.set_xlabel('T2M_RANGE (°C)')
ax.set_ylabel('WS2M (m/s)')
ax.set_title(f'{COUNTRY} — Diurnal Temperature Range vs. Wind Speed')
ax.grid(True)
plt.tight_layout()
plt.show()

---
## 6. Distribution Analysis

### 6.1 Precipitation Distribution (Log Scale)

In [ ]:
fig = plot_precipitation_histogram(df_clean, COUNTRY)
plt.show()

**📝 Distribution Shape:**  
_(Fill in — e.g., "Highly right-skewed even on log scale, confirming that most days see little or no precipitation and rare extreme events dominate the tail. This is characteristic of semi-arid climates.")_

### 6.2 Bubble Chart: Temperature vs Humidity

In [ ]:
fig = plot_bubble_chart(df_clean, COUNTRY)
plt.show()

---
## 7. Summary & COP32 Implications

### Key Findings

| Finding | Evidence | COP32 Demand |
|---------|----------|--------------|
| Temperature trend | _(insert trend value)_ | Adaptation finance for agriculture |
| Drought frequency | _(insert dry day count)_ | Early warning systems |
| Extreme rainfall | _(insert event count)_ | Flood resilience infrastructure |

### Negotiation-Grade Statement Draft
> *"Ethiopia's mean temperature has increased by X°C over the 2015–2026 period, with [year] recording the hottest [month] in the observational record. This warming trend directly threatens the livelihoods of 70 million Ethiopians dependent on rain-fed agriculture. The data demands a binding commitment to $X billion in adaptation finance and deployment of sub-national early warning systems before COP33."*

---
*Notebook complete. Cleaned file saved to `data/sudan_clean.csv`.*